# 1. Título

# 2. Membros (nome e número de matrícula)

- Daniel da Cunha Costa - 2024006064  
- Isaac Reyes Alves de Abreu - 2025050342  
- Pedro Luiz Siva - 2024006129

# 3. Descrição dos dados

O dataset que escolhemos trata sobre o comércio exterior brasileiro de bens por NCM (Nomenclatura Comum do Mercosul):
[Dados de comércio exterior brasileiro de bens](https://dados.gov.br/dados/conjuntos-dados/estatisticos-do-comercio-exterior-brasileiro-de-bens).  

Sua cobertura abrange os dados do ano de 2024(JAN - DEZ 2024) e contém todos os produtos (Nomenclatura Comum do Mercosul) que chegaram em alguma posto aduaneiro do Brasil (URF), além de informar de qual país veio a importação, qual o peso, o valor(FOB, frete e seguro) em U$, a cidade de destino, o mês de chegada e a quantidade estátistica.  

Juntamente com os dataset base, a fim de enriquecer os dados, foram baixadas tabelas adicionais, as quais foram essenciais para a realização dos joins e para dar significado ao código desenvolvido.

A fim de sondar a base de dados e garantir a corrude da análise futura, foi realizada a análise exploratória dos dados (EDA) usando Python + Pandas + Sqlite3, a fim de verificar a consistência dos dados e integridade do arquivo csv. Felizmente, o dataset estava em ótimo estado, bem documentando e completamente registrado, restando apenas enriquecê-lo e fazer uma limpeza de formatação. 

Conjuntamente, querys SQL foram realizadas para aumentar o entendimento da estrutura geral dos dados, seu comportamento e suas principais tendências, o que ajudou a orientar as consultas SQL posteriores.

Depois da Análise Exploratória dos Dados, já com os arquivos .csv limpos, realizou-se a migração dos dados para para o modelo relacional normalizado. Adicionalmente, algumas alterações nos dados base foram necessárias, tendo em vista atender as especificações, como a remoção de colunas desnecessárias e redundantes.


# 4. Diagrama ER

![Diagrama ER](esquemaER.jpg)

# 5. Diagrama relacional

![Esquema Relacional](relacional.jpg)

# 6. Consultas

In [5]:
import pandas as pd
import numpy as np
import sqlite3
import warnings
from pathlib import Path
import json
from datetime import datetime
import sys

# Visualização Estática
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches

# Visualização Interativa
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

# Análise Estatística Avançada
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Interface Interativa
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Configurações Globais
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Configuração Visual Avançada
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Configuração Matplotlib para Gráficos de Alta Qualidade
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

# Configuração Plotly para Interatividade
import plotly.io as pio
pio.templates.default = "plotly_white"

# Paletas de Cores Customizadas para Visualizações
PALETA_CORES = {
    'primaria': ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'],
    'sequencial': px.colors.sequential.Viridis,
    'divergente': px.colors.diverging.RdBu,
    'qualitativa': px.colors.qualitative.Set1,
    'paises': px.colors.qualitative.Dark24,
    'temporal': px.colors.sequential.Blues
}

In [6]:
# Configuração da Conexão ao Banco
DB_PATH = 'importacoes_brasil_2024.db'

class DatabaseManager:
    
    def __init__(self, db_path):
        self.db_path = db_path
        self.connection = None
        self.connect()
    
    def connect(self):
        """Estabelece conexão com SQLite"""
        try:
            self.connection = sqlite3.connect(self.db_path)
            # Otimizações para leitura
            self.connection.execute('PRAGMA journal_mode = WAL')
            self.connection.execute('PRAGMA synchronous = NORMAL')
            self.connection.execute('PRAGMA cache_size = 1000000')
            self.connection.execute('PRAGMA temp_store = MEMORY')
        except Exception as e:
            print(f"Erro ao conectar ao banco: {e}")
            raise
    
    def execute_query(self, query, params=None, return_df=True):
        try:
            if return_df:
                df = pd.read_sql_query(query, self.connection, params=params)
                return df
            else:
                cursor = self.connection.cursor()
                cursor.execute(query, params or ())
                return cursor.fetchall()
        except Exception as e:
            print(f"Erro na consulta: {e}")
            print(f"Query: {query[:100]}...")
            return pd.DataFrame() if return_df else []
    
    def get_table_info(self, table_name):
        query = f"PRAGMA table_info({table_name})"
        return self.execute_query(query)
    
    def get_tables(self):
        query = "SELECT name FROM sqlite_master WHERE type='table'"
        tables = self.execute_query(query, return_df=False)
        return [table[0] for table in tables]
    
    def get_row_count(self, table_name):
        query = f"SELECT COUNT(*) as count FROM {table_name}"
        result = self.execute_query(query)
        return result.iloc[0]['count'] if not result.empty else 0
    
    def close(self):
        if self.connection:
            self.connection.close()
            print("Conexão fechada")

db = DatabaseManager(DB_PATH)

if Path(DB_PATH).exists():
    tables = db.get_tables()
    print(f"\nTabelas disponíveis ({len(tables)}):")
    for i, table in enumerate(tables, 1):
        count = db.get_row_count(table)
        print(f"   {i:2d}. {table:25} ({count:>10,} registros)")
    
    def sql(query, params=None):
        return db.execute_query(query, params)
    
else:
    print(f"Banco não encontrado: {DB_PATH}")


Tabelas disponíveis (10):
    1. Unidade                   (        13 registros)
    2. Mes                       (        12 registros)
    3. sqlite_sequence           (         3 registros)
    4. UF                        (        27 registros)
    5. NCM                       (    13,721 registros)
    6. Pais                      (       281 registros)
    7. URF                       (       279 registros)
    8. Via                       (        17 registros)
    9. NCM_Unidade               (    13,721 registros)
   10. Importacoes               ( 2,273,687 registros)


## 6.1 Duas consultas envolvendo seleção e projeção

### 6.1.1 Consulta 1

In [32]:
def format_currency(value):
    """Formata valores monetários em bilhões, milhões ou milhares"""
    if value >= 1e9:
        return f"US$ {value/1e9:.1f}B"
    elif value >= 1e6:
        return f"US$ {value/1e6:.1f}M"
    elif value >= 1e3:
        return f"US$ {value/1e3:.1f}K"
    else:
        return f"US$ {value:.0f}"

query_1_1 = """
SELECT SUM(VL_FOB) as total_value 
FROM Importacoes;
"""

df_1_1 = db.execute_query(query_1_1, return_df=True)
display(df_1_1)


,total_value
0,262869305000


### 6.1.2 Consulta 2

In [34]:
query_1_2 = """
SELECT COUNT(DISTINCT COD_NCM) as ncms 
FROM Importacoes;
"""
df_1_2 = db.execute_query(query_1_2, return_df=True)
display(df_1_2)


,ncms
0,8748


## 6.2 Três consultas envolvendo junção de duas relações

### 6.2.1 Consulta 3: Evolução temporal das importações

In [8]:
query_2_1 = """
SELECT 
    m.NOME_MES,
    m.COD_MES,
    COUNT(*) as total_operacoes,
    SUM(i.VL_FOB) as valor_total,
    SUM(i.KG_LIQUIDO) as peso_total,
    AVG(i.VL_FOB) as valor_medio
FROM Importacoes i
JOIN Mes m ON i.COD_MES = m.COD_MES
GROUP BY m.COD_MES, m.NOME_MES
ORDER BY m.COD_MES;
"""
try:
    df_2_1 = db.execute_query(query_2_1, return_df=True)
    if not df_2_1.empty:
        # Gráfico de linhas com múltiplas métricas
        fig = make_subplots(
            rows=2, cols=1,
            subplot_titles=('Valor Total e Operações por Mês', 'Valor Médio por Operação'),
            specs=[[{"secondary_y": True}], [{"secondary_y": False}]]
        )
        
        # Valor total (linha)
        fig.add_trace(
            go.Scatter(x=df_2_1['NOME_MES'], y=df_2_1['valor_total'], 
                      mode='lines+markers', name='Valor Total', 
                      line=dict(color='blue', width=3)),
            row=1, col=1
        )
        
        # Número de operações (barras)
        fig.add_trace(
            go.Bar(x=df_2_1['NOME_MES'], y=df_2_1['total_operacoes'], 
                   name='Operações', marker_color='lightblue', opacity=0.7),
            row=1, col=1, secondary_y=True
        )
        
        # Valor médio
        fig.add_trace(
            go.Scatter(x=df_2_1['NOME_MES'], y=df_2_1['valor_medio'],
                      mode='lines+markers', name='Valor Médio',
                      line=dict(color='red', width=2)),
            row=2, col=1
        )
        
        fig.update_layout(
            title="<b>Consulta 2.1: Evolução Temporal das Importações</b>",
            height=700,
            template="plotly_white"
        )
        fig.update_xaxes(tickangle=45)
        fig.show()
        display(df_2_1)
        print(f"Resultado: {len(df_2_1)} meses analisados")
    else:
        print("Sem dados para visualizar")
except Exception as e:
    print(f"Erro: {e}")

,NOME_MES,COD_MES,total_operacoes,valor_total,peso_total,valor_medio
0,Janeiro,1,183979,20506553165,14271017701,111461.38
1,Fevereiro,2,177253,18217840718,11934056707,102778.74
2,Março,3,179226,20490209586,14131530601,114326.10
3,Abril,4,194280,21896274485,14996216363,112704.73
4,Maio,5,182010,21888476596,15314763004,120259.75
5,Junho,6,182202,22403496315,15987102383,122959.66
6,Julho,7,195553,23289874359,16852324456,119097.50
7,Agosto,8,195362,24219210009,17627190679,123970.94
8,Setembro,9,193426,23391780967,17357163568,120934.01
9,Outubro,10,201635,25209597414,18626111080,125025.90


Resultado: 12 meses analisados


### 6.2.2 Consulta 4: Top 20 países por valor de importação

In [9]:
query_2_2 = """
SELECT 
    p.NOME_PAIS,
    p.COD_PAIS,
    COUNT(*) as total_operacoes,
    SUM(i.VL_FOB) as valor_total,
    SUM(i.KG_LIQUIDO) as peso_total,
    AVG(i.VL_FOB) as valor_medio,
    SUM(i.VL_FRETE) as frete_total,
    SUM(i.VL_SEGURO) as seguro_total
FROM Importacoes i
JOIN Pais p ON i.COD_PAIS = p.COD_PAIS
GROUP BY p.COD_PAIS, p.NOME_PAIS
ORDER BY valor_total DESC
LIMIT 20;
"""
try:
    df_2_2 = db.execute_query(query_2_2, return_df=True)
    if not df_2_2.empty:
        # Treemap para mostrar proporções
        fig = go.Figure(go.Treemap(
            labels=df_2_2['NOME_PAIS'],
            values=df_2_2['valor_total'],
            parents=[""] * len(df_2_2),
            textinfo="label+value+percent parent",
            texttemplate="<b>%{label}</b><br>%{value}<br>%{percentParent}",
            hovertemplate='<b>%{label}</b><br>' +
                         'Valor Total: %{value:,.0f}<br>' +
                         'Operações: %{customdata[0]:,}<br>' +
                         'Peso Total: %{customdata[1]:,.0f} kg<br>' +
                         '<extra></extra>',
            customdata=list(zip(df_2_2['total_operacoes'], df_2_2['peso_total']))
        ))
        
        fig.update_layout(
            title="<b>Consulta 2.2: Top 20 Países por Valor de Importação</b>",
            height=600,
            template="plotly_white"
        )
        fig.show()
        display(df_2_2)
        print(f"Resultado: Top {len(df_2_2)} países analisados")
    else:
        print("Sem dados para visualizar")
except Exception as e:
    print(f"Erro: {e}")

,NOME_PAIS,COD_PAIS,total_operacoes,valor_total,peso_total,valor_medio,frete_total,seguro_total
0,China,160,510017,63636339238,26737510085,124772.98,5559494260,54165373
1,Estados Unidos,249,239654,40652245607,32327722315,169628.91,1805053670,26136891
2,Alemanha,23,179100,13783197105,1968668072,76958.11,390679340,16454569
3,Argentina,63,28713,13577045088,11044038950,472853.59,495694158,10715010
4,Rússia,676,2623,10965470014,23337603811,4180507.06,1221480596,21670463
5,Índia,361,76095,6849930671,2123427413,90018.14,413327987,4082459
6,Itália,386,130760,6388220214,844880025,48854.54,201129016,6164844
7,França,275,80580,6189829300,743615019,76815.95,125532759,4853973
8,México,493,50983,5767276053,728495280,113121.55,239059223,2966715
9,Japão,399,88525,5430941672,896919883,61349.24,236825939,5086782


Resultado: Top 20 países analisados


### 6.2.3 Consulta 5: Top 15 estados por valor de importação

In [30]:
query_2_3 = """
SELECT 
    uf.NOME_UF,
    uf.SIGLA_UF,
    COUNT(*) as total_operacoes,
    SUM(i.VL_FOB) as valor_total,
    SUM(i.KG_LIQUIDO) as peso_total,
    AVG(i.VL_FOB) as valor_medio
FROM Importacoes i
JOIN UF uf ON i.COD_UF = uf.COD_UF
GROUP BY uf.COD_UF, uf.NOME_UF, uf.SIGLA_UF
ORDER BY valor_total DESC;
"""
try:
    df_2_3 = db.execute_query(query_2_3, return_df=True)
    if not df_2_3.empty:
        # Gráfico de barras horizontais para todos os estados
        top_15 = df_2_3.head(15)
        
        fig = go.Figure(data=[
            go.Bar(
                y=top_15['SIGLA_UF'][::-1],  # Inverter para maior no topo
                x=top_15['valor_total'][::-1],
                orientation='h',
                marker=dict(
                    color=top_15['valor_total'][::-1],
                    colorscale='Blues',
                    showscale=True
                ),
                text=[format_currency(v) for v in top_15['valor_total'][::-1]],
                textposition='auto',
                hovertemplate='<b>%{y}</b><br>' +
                             'Valor: %{text}<br>' +
                             'Operações: %{customdata:,}<br>' +
                             '<extra></extra>',
                customdata=top_15['total_operacoes'][::-1]
            )
        ])
        
        fig.update_layout(
            title="<b>Consulta 2.3: Top 15 Estados por Valor de Importação</b>",
            xaxis_title="Valor Total (US$)",
            yaxis_title="Estado (UF)",
            height=600,
            template="plotly_white"
        )
        fig.show()
        display(df_2_3)
        print(f"Resultado: {len(df_2_3)} estados analisados")
    else:
        print("Sem dados para visualizar")
except Exception as e:
    print(f"Erro: {e}")

,NOME_UF,SIGLA_UF,total_operacoes,valor_total,peso_total,valor_medio
0,São Paulo,SP,788686,75882406908,24310016723,96213.71
1,Santa Catarina,SC,329079,33771587792,17484069079,102624.56
2,Rio de Janeiro,RJ,164006,27934201684,16388958562,170324.27
3,Paraná,PR,195820,19594722368,17993447994,100064.97
4,Minas Gerais,MG,199819,17016100064,13686266729,85157.57
5,Amazonas,AM,111553,16135054250,4754284102,144640.25
6,Espírito Santo,ES,58066,13886945704,9486424724,239157.95
7,Rio Grande do Sul,RS,136912,12980704170,14938574381,94810.57
8,Bahia,BA,43018,10675132111,15603138412,248155.01
9,Pernambuco,PE,43988,7440218641,6793890078,169142.01


Resultado: 27 estados analisados


## 6.3 Três consultas envolvendo junção de três ou mais relações

### 6.3.1 Consulta 6: Análise de Produtos (Valor × Operações × Peso)

In [12]:
query_3_3 = """
SELECT 
    n.COD_NCM,
    n.NOME_NCM,
    u.NOME_UNID,
    u.SIGLA_UNID,
    COUNT(*) as total_operacoes,
    SUM(i.VL_FOB) as valor_total,
    SUM(i.KG_LIQUIDO) as peso_total,
    SUM(i.QT_ESTATISTICA) as quantidade_total,
    AVG(i.VL_FOB) as valor_medio
FROM Importacoes i
JOIN NCM n ON i.COD_NCM = n.COD_NCM
JOIN Unidade u ON i.COD_UNID = u.COD_UNID
GROUP BY n.COD_NCM, n.NOME_NCM, u.NOME_UNID, u.SIGLA_UNID
ORDER BY valor_total DESC
LIMIT 20;
"""
try:
    df_3_3 = db.execute_query(query_3_3, return_df=True)
    if not df_3_3.empty:
        # Bubble chart - valor vs operações, tamanho = peso
        # Truncar nomes muito longos para visualização
        df_3_3['nome_curto'] = df_3_3['NOME_NCM'].str[:40] + '...'
        
        fig = go.Figure(data=go.Scatter(
            x=df_3_3['total_operacoes'],
            y=df_3_3['valor_total'],
            mode='markers',
            marker=dict(
                size=df_3_3['peso_total'],
                sizemode='area',
                sizeref=2.*max(df_3_3['peso_total'])/(40.**2),
                sizemin=4,
                color=df_3_3['valor_medio'],
                colorscale='Plasma',
                showscale=True,
                colorbar=dict(title="Valor Médio"),
                line=dict(width=2, color='DarkSlateGrey')
            ),
            text=df_3_3['nome_curto'],
            hovertemplate='<b>%{text}</b><br>' +
                         'Operações: %{x:,}<br>' +
                         'Valor Total: %{y:,.0f} USD<br>' +
                         'Peso Total: %{marker.size:,.0f} kg<br>' +
                         'Unidade: %{customdata}<br>' +
                         '<extra></extra>',
            customdata=df_3_3['SIGLA_UNID']
        ))
        
        fig.update_layout(
            title="<b>Consulta 3.1: Análise de Produtos (Valor × Operações × Peso)</b>",
            xaxis_title="Total de Operações",
            yaxis_title="Valor Total (US$)",
            height=600,
            template="plotly_white"
        )
        fig.show()
        display(df_3_3)
        print(f"Resultado: Top {len(df_3_3)} produtos analisados")
    else:
        print("Sem dados para visualizar")
except Exception as e:
    print(f"Erro: {e}")

,COD_NCM,NOME_NCM,NOME_UNID,SIGLA_UNID,total_operacoes,valor_total,peso_total,quantidade_total,valor_medio,nome_curto
0,27090010,Óleos brutos de petróleo,METRO CUBICO,M3,152,8690210062,13944653402,17791543,57172434.62,Óleos brutos de petróleo...
1,27101921,Gasóleo (óleo diesel),METRO CUBICO,M3,258,8359492092,12028127218,27554518,32401132.14,Gasóleo (óleo diesel)...
2,84119100,Partes de turborreatores ou de turbopropulsores,QUILOGRAMA LIQUIDO,KGL,1271,4837676499,310107,310107,3806197.09,Partes de turborreatores ou de turboprop...
3,31042090,Outros cloretos de potássio,QUILOGRAMA LIQUIDO,KGL,786,3606983824,13651091870,13651091870,4589037.94,Outros cloretos de potássio...
4,84111200,Turborreatores de empuxo superior a 25 kN,NUMERO (UNIDADE),UNID.,94,3418099488,2351149,1165,36362760.51,Turborreatores de empuxo superior a 25 k...
5,30021590,"Outros produtos imunológicos, apresentados em ...",QUILOGRAMA LIQUIDO,KGL,1056,3320554118,666234,666234,3144464.13,"Outros produtos imunológicos, apresentad..."
6,87042190,"Outros veículos automóveis com motor diesel, p...",NUMERO (UNIDADE),UNID.,192,3264984879,249046422,114862,17005129.58,Outros veículos automóveis com motor die...
7,27011200,"Hulha betuminosa, não aglomerada",QUILOGRAMA LIQUIDO,KGL,153,2837201476,15587414736,15587414736,18543800.50,"Hulha betuminosa, não aglomerada..."
8,31021010,"Ureia, mesmo em solução aquosa, com teor de ni...",QUILOGRAMA LIQUIDO,KGL,985,2687692092,8309735040,8309735040,2728621.41,"Ureia, mesmo em solução aquosa, com teor..."
9,27101241,Naftas para petroquimica,METRO CUBICO,M3,49,2628440404,3569294474,5197331,53641640.90,Naftas para petroquimica...


Resultado: Top 20 produtos analisados


### 6.3.2 Consulta 7

In [16]:
query_3_2 = """
SELECT 
    i.VL_FOB,
    i.KG_LIQUIDO,
    i.QT_ESTATISTICA,
    p.NOME_PAIS,
    n.NOME_NCM,
    uf.NOME_UF
FROM Importacoes i
JOIN Pais p ON i.COD_PAIS = p.COD_PAIS
JOIN NCM n ON i.COD_NCM = n.COD_NCM
JOIN UF uf ON i.COD_UF = uf.COD_UF
ORDER BY i.VL_FOB DESC
LIMIT 100;
"""
try:
    df_3_2 = db.execute_query(query_3_2, return_df=True)
    if not df_3_2.empty:
        # Sunburst chart para mostrar hierarquia País > UF > Produto
        # Preparar dados para sunburst
        df_sample = df_3_2.head(50)  # Top 50 para evitar sobrecarga visual
        
        # Criar hierarquia: País -> UF -> Produto (truncado)
        labels = []
        parents = []
        values = []
        
        # Nível 1: Países
        paises_unicos = df_sample['NOME_PAIS'].unique()
        for pais in paises_unicos:
            labels.append(pais)
            parents.append("")
            values.append(df_sample[df_sample['NOME_PAIS'] == pais]['VL_FOB'].sum())
        
        # Nível 2: UF por país
        for pais in paises_unicos:
            ufs_pais = df_sample[df_sample['NOME_PAIS'] == pais]['NOME_UF'].unique()
            for uf in ufs_pais:
                labels.append(f"{uf}")
                parents.append(pais)
                values.append(df_sample[(df_sample['NOME_PAIS'] == pais) & 
                                       (df_sample['NOME_UF'] == uf)]['VL_FOB'].sum())
        
        fig = go.Figure(go.Sunburst(
            labels=labels,
            parents=parents,
            values=values,
            branchvalues="total",
            hovertemplate='<b>%{label}</b><br>Valor: %{value:,.0f}<br><extra></extra>',
            maxdepth=3
        ))
        
        fig.update_layout(
            title="<b>Consulta 3.2: Outliers - Hierarquia País → UF → Valor</b>",
            height=600,
            template="plotly_white"
        )
        fig.show()
        
        # Gráfico adicional: Scatter dos outliers
        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(
            x=df_sample['KG_LIQUIDO'],
            y=df_sample['VL_FOB'],
            mode='markers',
            marker=dict(
                size=10,
                color=list(range(len(df_sample))),
                colorscale='Turbo',
                showscale=True,
                colorbar=dict(title="Ranking")
            ),
            text=df_sample['NOME_PAIS'],
            hovertemplate='<b>País:</b> %{text}<br>' +
                         '<b>Valor FOB:</b> %{y:,.0f} USD<br>' +
                         '<b>Peso:</b> %{x:,.0f} kg<br>' +
                         '<b>UF:</b> %{customdata}<br>' +
                         '<extra></extra>',
            customdata=df_sample['NOME_UF']
        ))
        
        fig2.update_layout(
            title="<b>Outliers: Valor FOB × Peso Líquido</b>",
            xaxis_title="Peso Líquido (kg)",
            yaxis_title="Valor FOB (US$)",
            height=500,
            template="plotly_white"
        )
        fig2.show()
        display(df_3_2)
        print(f"Resultado: Top {len(df_3_2)} outliers analisados")
    else:
        print("Sem dados para visualizar")
except Exception as e:
    print(f"Erro: {e}")

,VL_FOB,KG_LIQUIDO,QT_ESTATISTICA,NOME_PAIS,NOME_NCM,NOME_UF
0,545738786,43444272,24069,China,"Outros veículos, equipados para propulsão, sim...",Espírito Santo
1,418544007,275387066,275387066,Estados Unidos,Gás natural liquefeito,Bahia
2,282214052,150714,39,Estados Unidos,Turborreatores de empuxo superior a 25 kN,Rio de Janeiro
3,280357244,439677267,513254,Arábia Saudita,Óleos brutos de petróleo,Rio de Janeiro
4,277230470,158539,44,Estados Unidos,Turborreatores de empuxo superior a 25 kN,Rio de Janeiro
5,268385540,22623474,15096,China,"Outros veículos, equipados unicamente com moto...",Espírito Santo
6,264539557,10337,10337,Estados Unidos,Partes de turborreatores ou de turbopropulsores,Rio de Janeiro
7,260704510,11787,11787,Estados Unidos,Partes de turborreatores ou de turbopropulsores,Rio de Janeiro
8,253827275,10134,10134,Estados Unidos,Partes de turborreatores ou de turbopropulsores,Rio de Janeiro
9,249968167,154582,38,Estados Unidos,Turborreatores de empuxo superior a 25 kN,Rio de Janeiro


Resultado: Top 100 outliers analisados


### 6.3.3 Consulta 8: Heatmap temporal - Top 10 países x meses

In [19]:
query_3_3 = """
SELECT 
    p.NOME_PAIS,
    m.NOME_MES,
    SUM(i.VL_FOB) as valor_total
FROM Importacoes i
JOIN Pais p ON i.COD_PAIS = p.COD_PAIS
JOIN Mes m ON i.COD_MES = m.COD_MES
WHERE UPPER(p.NOME_PAIS) IN ('CHINA', 'ESTADOS UNIDOS', 'ALEMANHA', 'ARGENTINA', 'COREIA DO SUL', 'ÍNDIA', 'ITÁLIA', 'FRANÇA', 'JAPÃO', 'CHILE')
GROUP BY p.NOME_PAIS, m.NOME_MES, m.COD_MES
ORDER BY m.COD_MES;
"""
try:
    df_3_3 = sql(query_3_3)
    if not df_3_3.empty:
        # Criar matriz pivot para heatmap
        heatmap_matrix = df_3_3.pivot(index='NOME_PAIS', columns='NOME_MES', values='valor_total')
        heatmap_matrix = heatmap_matrix.fillna(0)
        
        # Ordenar colunas por ordem dos meses
        meses_ordem = ['Janeiro', 'Fevereiro', 'Março', 'Abril', 'Maio', 'Junho',
                       'Julho', 'Agosto', 'Setembro', 'Outubro', 'Novembro', 'Dezembro']
        colunas_disponiveis = [mes for mes in meses_ordem if mes in heatmap_matrix.columns]
        if colunas_disponiveis:
            heatmap_matrix = heatmap_matrix[colunas_disponiveis]
        
        # Heatmap interativo
        fig = go.Figure(data=go.Heatmap(
            z=heatmap_matrix.values,
            x=heatmap_matrix.columns,
            y=heatmap_matrix.index,
            colorscale=PALETA_CORES['sequencial'],
            showscale=True,
            colorbar=dict(title="Valor (US$)"),
            hovertemplate='<b>País:</b> %{y}<br>' +
                         '<b>Mês:</b> %{x}<br>' +
                         '<b>Valor:</b> %{customdata}<br>' +
                         '<extra></extra>',
            customdata=[[format_currency(v) for v in row] for row in heatmap_matrix.values]
        ))
        
        fig.update_layout(
            title="<b>Consulta 3.3: Heatmap Temporal - Top 10 Países × Meses</b>",
            height=500,
            template="plotly_white"
        )
        fig.show()
        display(df_3_3)
        print(f"Resultado: {len(heatmap_matrix)} países × {len(heatmap_matrix.columns)} meses")
    else:
        print("❌ Sem dados para visualizar. Verifique os nomes dos países na base de dados.")
except Exception as e:
    print(f"❌ Erro: {e}")

,NOME_PAIS,NOME_MES,valor_total
0,Alemanha,Janeiro,1096014455
1,Argentina,Janeiro,796542328
2,Chile,Janeiro,352203614
3,China,Janeiro,5059011101
4,Coreia do Sul,Janeiro,407981516
5,Estados Unidos,Janeiro,3199689284
6,Índia,Janeiro,443174556
7,Alemanha,Fevereiro,1023715880
8,Argentina,Fevereiro,717274011
9,Chile,Fevereiro,357390892


Resultado: 7 países × 12 meses


## 6.4 Duas consultas envolvendo agregação sobre junção de duas ou mais relações

### 6.4.1 Consulta 9

In [27]:
query_4_1 = """
SELECT p.COD_PAIS, p.NOME_PAIS, SUM(i.VL_FOB) as total_value
FROM Importacoes i
JOIN Pais p ON i.COD_PAIS = p.COD_PAIS
GROUP BY p.COD_PAIS, p.NOME_PAIS
ORDER BY total_value DESC
LIMIT 20;
"""

try:
    df_4_1 = db.execute_query(query_4_1, return_df=True)
    if not df_4_1.empty:
        # Gráfico de barras para os 20 principais países
        fig = go.Figure(data=go.Bar(
            x=df_4_1['NOME_PAIS'],
            y=df_4_1['total_value'],
            marker_color='indianred',
            text=[format_currency(v) for v in df_4_1['total_value']],
            textposition='auto',
            hovertemplate='<b>%{x}</b><br>' +
                         'Valor Total: %{y:,.0f} USD<br>' +
                         '<extra></extra>'
        ))
        
        fig.update_layout(
            title="<b>Consulta 4.1: Top 20 Países por Valor de Importação</b>",
            xaxis_title="País",
            yaxis_title="Valor Total (US$)",
            height=600,
            template="plotly_white"
        )
        fig.show()
        display(df_4_1)
        print(f"Resultado: {len(df_4_1)} países analisados")
    else:
        print("Sem dados para visualizar")
except Exception as e:
    print(f"Erro: {e}")

,COD_PAIS,NOME_PAIS,total_value
0,160,China,63636339238
1,249,Estados Unidos,40652245607
2,23,Alemanha,13783197105
3,63,Argentina,13577045088
4,676,Rússia,10965470014
5,361,Índia,6849930671
6,386,Itália,6388220214
7,275,França,6189829300
8,493,México,5767276053
9,399,Japão,5430941672


Resultado: 20 países analisados


### 6.4.2 Consulta 10: Análise de eficiencia logística - URF x País x Via

In [26]:
query_4_2 = """
SELECT 
    u.NOME_URF,
    p.NOME_PAIS,
    v.NOME_VIA,
    COUNT(*) as total_operacoes,
    SUM(i.VL_FOB) as valor_total,
    AVG(i.VL_FOB) as valor_medio,
    SUM(i.KG_LIQUIDO) as peso_total,
    AVG(i.KG_LIQUIDO) as peso_medio,
    SUM(i.VL_FRETE) as frete_total,
    AVG(i.VL_FRETE) as frete_medio,
    -- Indicadores de eficiência
    ROUND(SUM(i.VL_FRETE) / NULLIF(SUM(i.VL_FOB), 0) * 100, 2) as percentual_frete,
    ROUND(SUM(i.VL_FOB) / NULLIF(SUM(i.KG_LIQUIDO), 0), 2) as valor_por_kg
FROM Importacoes i
JOIN URF u ON i.COD_URF = u.COD_URF
JOIN Pais p ON i.COD_PAIS = p.COD_PAIS  
JOIN Via v ON i.COD_VIA = v.COD_VIA
WHERE i.VL_FOB > 0 AND i.KG_LIQUIDO > 0
GROUP BY u.COD_URF, u.NOME_URF, p.COD_PAIS, p.NOME_PAIS, v.COD_VIA, v.NOME_VIA
HAVING COUNT(*) >= 10
ORDER BY valor_total DESC
LIMIT 30;
"""

try:
    df_4_2 = db.execute_query(query_4_2, return_df=True)
    if not df_4_2.empty:
        
        fig = go.Figure()
        
        df_4_2['URF_curto'] = df_4_2['NOME_URF'].str[:20] + '...'
        df_4_2['hover_text'] = df_4_2['URF_curto'] + ' - ' + df_4_2['NOME_PAIS'] + ' (' + df_4_2['NOME_VIA'] + ')'
        
        fig.add_trace(go.Scatter(
            x=df_4_2['valor_por_kg'],
            y=df_4_2['valor_total'],
            mode='markers',
            marker=dict(
                size=df_4_2['total_operacoes'],
                sizemode='area',
                sizeref=2.*max(df_4_2['total_operacoes'])/(60.**2),
                sizemin=8,
                color=df_4_2['percentual_frete'],
                colorscale='RdYlBu_r',  # Vermelho = alto frete, Azul = baixo frete
                showscale=True,
                colorbar=dict(title="% Frete"),
                line=dict(width=1, color='DarkSlateGrey'),
                opacity=0.8
            ),
            text=df_4_2['hover_text'],
            hovertemplate='<b>%{text}</b><br>' +
                         'Valor por kg: %{x:,.2f} USD/kg<br>' +
                         'Valor Total: %{y:,.0f} USD<br>' +
                         'Operações: %{marker.size:,}<br>' +
                         'Frete: %{marker.color:.1f}% do valor<br>' +
                         'Peso Total: %{customdata[0]:,.0f} kg<br>' +
                         'Frete Médio: %{customdata[1]:,.0f} USD<br>' +
                         '<extra></extra>',
            customdata=list(zip(df_4_2['peso_total'], df_4_2['frete_medio']))
        ))
        
        fig.update_layout(
            title="<b>Consulta 4.2: Análise de Eficiência Logística (URF × País × Via)</b><br>" +
                  "<sub>Eixo X: Valor por kg | Eixo Y: Valor Total | Tamanho: Operações | Cor: % Frete</sub>",
            xaxis_title="Valor por kg (US$/kg) - Indicador de Valor Agregado",
            yaxis_title="Valor Total (US$)",
            height=700,
            template="plotly_white",
            showlegend=False
        )
        
        valor_medio_kg = df_4_2['valor_por_kg'].mean()
        fig.add_vline(x=valor_medio_kg, line_dash="dash", line_color="gray", 
                     annotation_text=f"Valor médio/kg: {valor_medio_kg:.2f}")
        
        valor_total_medio = df_4_2['valor_total'].mean()
        fig.add_hline(y=valor_total_medio, line_dash="dash", line_color="gray",
                     annotation_text=f"Valor total médio: {format_currency(valor_total_medio)}")
        
        fig.show()
        
        top_eficientes = df_4_2.nsmallest(15, 'percentual_frete')
        
        fig2 = go.Figure(data=go.Bar(
            x=top_eficientes['percentual_frete'],
            y=[f"{row['URF_curto']} - {row['NOME_VIA']}" for _, row in top_eficientes.iterrows()],
            orientation='h',
            marker=dict(
                color=top_eficientes['percentual_frete'],
                colorscale='RdYlGn_r',  # Vermelho = alto, Verde = baixo
                showscale=True,
                colorbar=dict(title="% Frete")
            ),
            text=[f"{val:.1f}%" for val in top_eficientes['percentual_frete']],
            textposition='auto',
            hovertemplate='<b>%{y}</b><br>' +
                         'Frete: %{x:.1f}% do valor<br>' +
                         'Valor Total: %{customdata[0]}<br>' +
                         'Operações: %{customdata[1]:,}<br>' +
                         '<extra></extra>',
            customdata=list(zip([format_currency(v) for v in top_eficientes['valor_total']], 
                               top_eficientes['total_operacoes']))
        ))
        
        fig2.update_layout(
            title="<b>Top 15 Rotas Mais Eficientes (Menor % de Frete)</b>",
            xaxis_title="Percentual do Frete sobre Valor FOB (%)",
            yaxis_title="URF - Via de Transporte",
            height=600,
            template="plotly_white",
            margin=dict(l=350),  # aumenta a margem esquerda
            yaxis=dict(automargin=True, tickfont=dict(size=14))  # garante espaço para o texto
        )
        fig2.show()
        
        resumo_via = df_4_2.groupby('NOME_VIA').agg({
            'total_operacoes': 'sum',
            'valor_total': 'sum',
            'peso_total': 'sum',
            'percentual_frete': 'mean',
            'valor_por_kg': 'mean'
        }).round(2).sort_values('valor_total', ascending=False)
        
        print(f"\nINSIGHTS DA ANÁLISE:")
        via_mais_movimentada = resumo_via.index[0]
        print(f"   Via mais movimentada: {via_mais_movimentada}")
        print(f"   Valor total: {format_currency(resumo_via.loc[via_mais_movimentada, 'valor_total'])}")
        
        rota_mais_eficiente = top_eficientes.iloc[0]
        print(f"   Rota mais eficiente: {rota_mais_eficiente['NOME_URF']} - {rota_mais_eficiente['NOME_VIA']}")
        print(f"   Frete: {rota_mais_eficiente['percentual_frete']:.1f}% do valor")
        
        maior_valor_kg = df_4_2.loc[df_4_2['valor_por_kg'].idxmax()]
        print(f"   Maior valor agregado: {maior_valor_kg['NOME_URF']} - {maior_valor_kg['NOME_PAIS']}")
        print(f"   Valor/kg: US$ {maior_valor_kg['valor_por_kg']:.2f}/kg")
        
        display(df_4_2)
        print(f"\nResultado: {len(df_4_2)} combinações URF×País×Via analisadas")
        
    else:
        print("Sem dados para visualizar")
except Exception as e:
    print(f"Erro: {e}")


INSIGHTS DA ANÁLISE:
   Via mais movimentada: MARITIMA
   Valor total: US$ 92.3B
   Rota mais eficiente: PORTO DE SANTOS - MARITIMA
   Frete: 0.0% do valor
   Maior valor agregado: AEROPORTO INTERNACIONAL DO RIO DE JANEIRO - Estados Unidos
   Valor/kg: US$ 968.00/kg


,NOME_URF,NOME_PAIS,NOME_VIA,total_operacoes,valor_total,valor_medio,peso_total,peso_medio,frete_total,frete_medio,percentual_frete,valor_por_kg,URF_curto,hover_text
0,PORTO DE SANTOS,China,MARITIMA,121655,21127651574,173668.58,6779120946,55724.15,1513258939,12438.94,0.00,3.00,PORTO DE SANTOS...,PORTO DE SANTOS... - China (MARITIMA)
1,PORTO DE SANTOS,Estados Unidos,MARITIMA,45675,9375825835,205272.60,4334939608,94908.37,316392509,6927.04,0.00,2.00,PORTO DE SANTOS...,PORTO DE SANTOS... - Estados Unidos (MARITIMA)
2,PORTO DE SAO FRANCISCO DO SUL,China,MARITIMA,41386,6722244127,162427.97,3598243104,86943.49,581886795,14059.99,0.00,1.00,PORTO DE SAO FRANCIS...,PORTO DE SAO FRANCIS... - China (MARITIMA)
3,ITAJAI,China,MARITIMA,41552,6681017464,160786.90,2420063385,58241.80,695263474,16732.37,0.00,2.00,ITAJAI...,ITAJAI... - China (MARITIMA)
4,AEROPORTO INTERNACIONAL DE VIRACOPOS,Estados Unidos,AEREA,43835,6064376314,138345.53,24108784,549.99,171400344,3910.13,0.00,251.00,AEROPORTO INTERNACIO...,AEROPORTO INTERNACIO... - Estados Unidos (AEREA)
5,PORTO DE SANTOS,Alemanha,MARITIMA,36207,5758718461,159049.86,922204788,25470.35,88635842,2448.03,0.00,6.00,PORTO DE SANTOS...,PORTO DE SANTOS... - Alemanha (MARITIMA)
6,PORTO DE PARANAGUA,China,MARITIMA,35100,5552168997,158181.45,4298089059,122452.68,442996449,12620.98,0.00,1.00,PORTO DE PARANAGUA...,PORTO DE PARANAGUA... - China (MARITIMA)
7,AEROPORTO INTERNACIONAL DO RIO DE JANEIRO,Estados Unidos,AEREA,12981,3878857213,298810.35,4004269,308.47,38000252,2927.37,0.00,968.00,AEROPORTO INTERNACIO...,AEROPORTO INTERNACIO... - Estados Unidos (AEREA)
8,PORTO DE MANAUS,China,MARITIMA,11815,3826597716,323876.23,908550435,76898.05,322910648,27330.57,0.00,4.00,PORTO DE MANAUS...,PORTO DE MANAUS... - China (MARITIMA)
9,PORTO DE VITORIA,China,MARITIMA,10631,3389521572,318833.75,827413628,77830.27,277666710,26118.59,0.00,4.00,PORTO DE VITORIA...,PORTO DE VITORIA... - China (MARITIMA)



Resultado: 30 combinações URF×País×Via analisadas


In [35]:
db.close()

Conexão fechada


# 7. Autoavaliação dos membros